# TTS Client
To be ran in conjunction with `tts__engine`

In [2]:
import websockets
import openai
import asyncio

In [3]:
client = openai.OpenAI(base_url="http://127.0.0.1:5000/v1", api_key="0608da5d28eb10cea2914f3de0f3ddba")

In [12]:
response = client.chat.completions.create(
    model="cognitivecomputations_dolphin-2.9-llama3-8b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "tell me a short story (2 sentences)"
        }
    ],
    max_tokens=350,
    stream=True
)


import time
time_start = 0
# openai_gen = openai_generator(response)
# Establish a connection to the websocket server
websocket = await websockets.connect("ws://localhost:8000/ws")
# websocket = await websockets.connect("ws://localhost:8000/api/v1/tts/ws")

async def feed_to_websocket(response):
    num_chunks = 0
    print("feed_to_websocket called")
    for chunk in response:
        print(chunk.choices[0].delta.content, end="")
        if chunk.choices[0].delta.content is None:
            break
        if ((num_chunks := num_chunks + 1) % 15) == 0:
            print("\n{:.2f}".format(time.time() - time_start), ":", num_chunks, "chunks sent")
            yield
            print("FTW")

        # Feed the chunk to the websocket
        await websocket.send(chunk.choices[0].delta.content)
         
    # Add a final END signal to the websocket
    await websocket.send("END")
    print("\n{:.2f}".format(time.time() - time_start), ":", "END sent")

async def receive_from_websocket(websocket, until_done=False):
    print("receive_from_websocket called")
    got_first_bunch = False
    try:
        counter = 0
        while (counter := counter + 1) % 100 != 0 or until_done:
            # Message is either bytes or text "END"
            message = await websocket.recv()
            if message == "END":
                break
            if counter % 100 == 0:
                if not got_first_bunch:
                    print("{:.2f}".format(time.time() - time_start), ":", "First chunk received")
                    got_first_bunch = True
                # Print time every 100 messages
                print("{:.2f}".format(time.time() - time_start), ":", "100 chunks received")
                # yield
                print("RFW")

            # counter += 1

        print("Recv->", counter)
        print("RECEIVED END")
    except websockets.exceptions.ConnectionClosedError:
        print("Connection closed")
    except Exception as e:
        print(e)

async def ws_thread_manager(response):
    is_done = False
    try:
        async for _ in feed_to_websocket(response):
            await receive_from_websocket(websocket)
        await receive_from_websocket(websocket, until_done=True)
    except websockets.exceptions.ConnectionClosedError:
        print("Connection closed")
    except Exception as e:
        print(e)
        
async def gen():
    global time_start
    time_start = time.time()
    # feeder = asyncio.create_task(feed_to_websocket(response))
    # await receive_from_websocket(websocket)
    manager = asyncio.create_task(ws_thread_manager(response))
    await asyncio.gather(manager)
    print("Time taken:", time.time() - time_start)
    
    # Close the websocket connection
    await websocket.close()


await gen()


# 

feed_to_websocket called
Once upon a time, there was a small village where everyone lived happily,
0.00 : 15 chunks sent
receive_from_websocket called
Recv-> 100
RECEIVED END
FTW
 caring for each other and their surroundings. One day, a magical bird appeared
0.01 : 30 chunks sent
receive_from_websocket called
Recv-> 100
RECEIVED END
FTW
, spreading joy and positivity wherever it went, ensuring that the villagers continued to
0.22 : 45 chunks sent
receive_from_websocket called
Recv-> 100
RECEIVED END
FTW
 live in harmony for generations to follow.None
0.56 : END sent
Done sending
receive_from_websocket called
0.56 : First chunk received
0.56 : 100 chunks received
RFW
0.56 : 100 chunks received
RFW
0.56 : 100 chunks received
RFW
0.56 : 100 chunks received
RFW
0.56 : 100 chunks received
RFW
0.59 : 100 chunks received
RFW
0.60 : 100 chunks received
RFW
1.13 : 100 chunks received
RFW
1.13 : 100 chunks received
RFW
1.67 : 100 chunks received
RFW
2.17 : 100 chunks received
RFW
2.17 : 100 chun

### Async but limited to one thread at a time

In [ ]:
response = client.chat.completions.create(
    model="cognitivecomputations_dolphin-2.9-llama3-8b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "tell me a short story (2 sentences)"
        }
    ],
    max_tokens=350,
    stream=True
)


import time
time_start = 0
# openai_gen = openai_generator(response)
# Establish a connection to the websocket server
# websocket = await websockets.connect("ws://localhost:8000/ws")
websocket = await websockets.connect("ws://localhost:8000/api/v1/tts/ws")

async def feed_to_websocket(response):
    for chunk in response:
        print(chunk.choices[0].delta.content, end="")
        if chunk.choices[0].delta.content is None:
            break
        # Feed the chunk to the websocket
        await websocket.send(chunk.choices[0].delta.content)
         
    # Add a final END signal to the websocket
    await websocket.send("END")
    print("\n{:.2f}".format(time.time() - time_start), ":", "END sent")

async def receive_from_websocket(websocket):
    print("receive_from_websocket called")
    got_first_bunch = False
    try:
        counter = 0
        while True:
            # Message is either bytes or text "END"
            message = await websocket.recv()
            if message == "END":
                break
            if counter % 100 == 0:
                if not got_first_bunch:
                    print("{:.2f}".format(time.time() - time_start), ":", "First chunk received")
                    got_first_bunch = True
                # Print time every 100 messages
                print("{:.2f}".format(time.time() - time_start), ":", "100 chunks received")
            counter += 1

        print("Recv->", counter)
        print("RECEIVED END")
    except websockets.exceptions.ConnectionClosedError:
        print("Connection closed")
    except Exception as e:
        print(e)
        
        
async def gen():
    global time_start
    time_start = time.time()
    feeder = asyncio.create_task(feed_to_websocket(response))
    await receive_from_websocket(websocket)
        
    await asyncio.gather(feeder)
    print("Time taken:", time.time() - time_start)
    
    # Close the websocket connection
    await websocket.close()


await gen()


# 

receive_from_websocket called
Once upon a midnight dreary, a brave knight ventured into a dark forest in search of a mysterious fairy who could grant him eternal youth. After facing many dangers and overcoming perilous obstacles, he eventually discovered the legendary fairy, who upon seeing his pure heart and undying bravery, bestowed upon him the gift of eternal youth.None
0.85 : END sent
0.86 : First chunk received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.86 : 100 chunks received
0.87 : 100 chunks received
0.87 : 100 chunks received
0.87 : 100 chunks received
0.87 : 100 chunks received
0.87 : 100 chunks received
0.87 : 100 chunks received
0.87 : 100 chunks received
0.87 : 100 chunks received
1.29 : 100 chunks received
1.82 : 100 c